# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hariommishra-12/Flyrank-Project/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions


The key traffic fields show strong right-skewed distributions. Impressions, clicks, sessions, and search volume have heavy tails, meaning a small number of pages account for much larger values than most pages. Because of these heavy tails, median and upper-percentile values are more useful than the mean alone when interpreting the data.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
import pandas as pd
import numpy as np

# Load repository
!git clone https://github.com/anujrkt06-tech/Flyrank-ML-project-.git

%cd Flyrank-ML-project-

# Load dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

# Key fields
key_fields = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "search_volume",
    "word_count"
]

# Distribution summary
summary = df[key_fields].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
).T

print("\nDistribution summary:")
print(summary.round(2))

# Skewness
print("\nSkewness:")
print(df[key_fields].skew().round(2))

Cloning into 'Flyrank-ML-project-'...
remote: Enumerating objects: 410, done.
remote: Counting objects: 100% (148/148), done.
remote: Compressing objects: 100% (146/146), done.
remote: Total 410 (delta 105), reused 2 (delta 2), pack-reused 262 (from 3)
Receiving objects: 100% (410/410), 2.01 MiB | 8.99 MiB/s, done.
Resolving deltas: 100% (236/236), done.
/content/Flyrank-ML-project-/Flyrank-ML-project-/Flyrank-ML-project-/Flyrank-ML-project-/Flyrank-ML-project-
Dataset loaded successfully!
Dataset shape: (30000, 44)

Distribution summary:
                   count     mean       std  min     50%      75%      90%  \
impressions_90d  30000.0  5200.37  16838.02  1.0   731.0  3615.25  12136.4   
clicks_90d       30000.0    16.10     75.08  0.0     1.0     7.00     32.0   
sessions_90d     30000.0    37.07    107.07  1.0     7.0    27.00     88.0   
search_volume    27532.0   158.88   1518.27  0.0    10.0    20.00    110.0   
word_count       22301.0  3107.76   1452.38  8.0  2877.0  3666.00


## 2. Signal test #1 / #2 / #3

Signal #1 — Search volume and impressions: The measured correlation is very weak, so search volume alone does not appear to be a strong indicator of 90-day impressions. Verdict: FALSE.

Signal #2 — Position and CTR: The measured CTR changes across position tiers, with deeper positions showing lower CTR. This supports the expected directional relationship. Verdict: CONFIRMED.

Signal #3 — Word count and trend: The measured median word counts for growing and declining pages are relatively similar. This does not provide strong evidence that word count alone explains trend direction. Verdict: FALSE.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Signal test #1: Search volume vs impressions

corr = df["search_volume"].corr(df["impressions_90d"])

print("Signal #1: Search volume vs impressions")
print("Correlation:", round(corr, 4))
print("Verdict: FALSE")


# Signal test #2: Position tier vs CTR

visible = df[df["impressions_90d"] >= 100].copy()

position_result = (
    visible
    .groupby("position_tier")
    .agg(
        pages=("content_id", "count"),
        impressions=("impressions_90d", "sum"),
        clicks=("clicks_90d", "sum")
    )
)

position_result["weighted_ctr"] = (
    position_result["clicks"]
    / position_result["impressions"]
    * 100
)

print("\nSignal #2: Position tier vs CTR")
print(position_result.round(3))
print("Verdict: CONFIRMED")


# Signal test #3: Word count vs trend

trend_words = (
    df[df["trend_direction"].isin(["up", "down"])]
    .groupby("trend_direction")["word_count"]
    .median()
)

print("\nSignal #3: Word count vs trend")
print(trend_words.round(0))

difference_pct = (
    abs(trend_words["up"] - trend_words["down"])
    / ((trend_words["up"] + trend_words["down"]) / 2)
) * 100

print("Median difference (%):", round(difference_pct, 2))
print("Verdict: FALSE")

Signal #1: Search volume vs impressions
Correlation: 0.0012
Verdict: FALSE

Signal #2: Position tier vs CTR
               pages  impressions  clicks  weighted_ctr
position_tier                                          
deep             879      1213203     479         0.039
page_1          8633     89493618  313097         0.350
page_3_5        6058     35141017   54370         0.155
striking        5903     22946217   79584         0.347
top_3            533      7025180   34222         0.487
Verdict: CONFIRMED

Signal #3: Word count vs trend
trend_direction
down    2909.0
up      2848.0
Name: word_count, dtype: float64
Median difference (%): 2.14
Verdict: FALSE



## 3. The flag-linked test

A flag-linked assumption is that pages with meaningful search visibility but weak CTR may need a CTR-related review. The measured CTR varies across position tiers, and deeper positions show lower CTR. The data therefore supports this assumption directionally, although position and traffic volume should also be considered before treating low CTR as a definite problem. Verdict: CONFIRMED, directionally.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Flag-linked test
# Test CTR behaviour without using the flag itself

test = df[df["impressions_90d"] >= 100].copy()

flag_test = (
    test
    .groupby("position_tier")
    .agg(
        pages=("content_id", "count"),
        impressions=("impressions_90d", "sum"),
        clicks=("clicks_90d", "sum")
    )
)

flag_test["weighted_ctr"] = (
    flag_test["clicks"]
    / flag_test["impressions"]
    * 100
)

print("Flag-linked test: CTR by position tier")
print(flag_test.round(3))

overall_ctr = (
    test["clicks_90d"].sum()
    / test["impressions_90d"].sum()
) * 100

print("\nOverall weighted CTR:", round(overall_ctr, 3), "%")

print("\nVerdict: CONFIRMED, directionally")

Flag-linked test: CTR by position tier
               pages  impressions  clicks  weighted_ctr
position_tier                                          
deep             879      1213203     479         0.039
page_1          8633     89493618  313097         0.350
page_3_5        6058     35141017   54370         0.155
striking        5903     22946217   79584         0.347
top_3            533      7025180   34222         0.487

Overall weighted CTR: 0.309 %

Verdict: CONFIRMED, directionally



## 4. What this means in practice

The content team should not rely on a single metric such as search volume or word count when making content decisions. Position and CTR provide a stronger directional signal, but traffic volume should also be considered. These findings are decision-support evidence and should not be treated as proof of causation.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Practical takeaway

print("Practical takeaway:")

print(
    "Use multiple measured signals together rather than relying "
    "on one metric."
)

print(
    "Position and CTR show a stronger directional relationship, "
    "but traffic volume should also be considered."
)

print(
    "These results are decision-support signals, not proof of causation."
)

Practical takeaway:
Use multiple measured signals together rather than relying on one metric.
Position and CTR show a stronger directional relationship, but traffic volume should also be considered.
These results are decision-support signals, not proof of causation.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.